# ROME workshop · clean notebook

**Dvě buňky.** První připraví větev `workshop-version`, druhá otevře Gradio nahoře.
**Odkaz na kovarianci vložte do pole `COVARIANCE_DRIVE_URL` v první buňce.**
Stávající odkaz je předvyplněný; ponechte ho nebo ho nahraďte vlastním Google Drive odkazem.
Druhá buňka jej předá do Gradio do pole **Google Drive odkaz na kovarianci**.
Tam klikněte na **Připravit model**; případně místo odkazu nahrajte kovarianci `.pt`.

> **Implementaci neupravujte.** Lokální implementace a větev jsou připravené
> v rámci workshopového PR. Není potřeba měnit Python kód, `LOCAL_LATIUM`, `LATIUM_REF`
> ani nastavovat vlastní checkout. Ponechte výchozí hodnoty a spusťte obě buňky shora dolů.
> Vlastní fakta zadáváte až do formuláře v Gradio; zdrojové panely slouží k nahlédnutí do kódu.

Qwen3.5-4B, ROME, causal tracing, GRAM i frontend se importují z Latium;
notebook žádnou z těchto implementací nekopíruje. Vyberte CUDA GPU runtime.
Model se načítá až tlačítkem v Gradio. Původní bílo-červeno-tyrkysový vzhled zůstává.


In [ ]:
#@title 1 · Připravit Latium workshop-version
# Přednastavené pro workshop/PR. Účastníci tyto vývojářské volby nemění.
LOCAL_LATIUM = ""
LATIUM_REF = "workshop-version"
AUTHENTICATE_DRIVE = False #@param {type:"boolean"}
SEED = 42 #@param {type:"integer"}
COVARIANCE_DRIVE_URL = "https://drive.google.com/file/d/1UmPlov94UKE4KIHclswVUMXyliQM3qvU/view?usp=drive_link" #@param {type:"string"}

import importlib.metadata
import subprocess
import sys
from pathlib import Path

sibling = Path.cwd().parent / "Latium-workshop-version"
repo = (Path(LOCAL_LATIUM).expanduser() if LOCAL_LATIUM else
        sibling if (sibling / ".git").exists() else Path.cwd() / "Latium-workshop-version").resolve()
if not repo.exists():
    subprocess.run(["git", "clone", "--depth=1", "--single-branch", "--branch", LATIUM_REF,
                    "https://github.com/Security-FIT/Latium.git", str(repo)], check=True)
if not (repo / "src/workshop/app.py").is_file():
    raise RuntimeError(f"{repo} neobsahuje workshop. Vyberte checkout workshop-version pomocí LOCAL_LATIUM.")
branch = subprocess.check_output(["git", "-C", str(repo), "branch", "--show-current"], text=True).strip()
if branch != LATIUM_REF:
    raise RuntimeError(f"Checkout je na větvi {branch!r}, očekává se {LATIUM_REF!r}. Existující práci nepřepínám.")
loaded_src = sys.modules.get("src")
if loaded_src is not None and not Path(loaded_src.__file__).resolve().is_relative_to(repo):
    raise RuntimeError("Kernel již importoval jiné Latium. Restartujte runtime a spusťte tento notebook znovu.")
if not LOCAL_LATIUM and repo != sibling.resolve() and loaded_src is None:
    origin = subprocess.check_output(["git", "-C", str(repo), "remote", "get-url", "origin"], text=True).strip()
    dirty = subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain"], text=True).strip()
    if origin in {"https://github.com/Security-FIT/Latium.git", "git@github.com:Security-FIT/Latium.git"} and not dirty:
        subprocess.run(["git", "-C", str(repo), "fetch", "origin", LATIUM_REF], check=True)
        subprocess.run(["git", "-C", str(repo), "merge", "--ff-only", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-workshop.txt")], check=True)
for package in ("torch", "transformers", "gradio"):
    loaded = sys.modules.get(package)
    if loaded is not None and loaded.__version__ != importlib.metadata.version(package):
        raise RuntimeError(f"{package} byl aktualizován po importu. Restartujte runtime a spusťte buňku znovu.")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
revision = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Latium {branch} · {revision} · {repo}")
if AUTHENTICATE_DRIVE:
    from google.colab import auth
    auth.authenticate_user()


In [ ]:
#@title 2 · Otevřít workshop
import inspect
from src.workshop import launch_workshop

if not {"seed", "covariance_drive_url"}.issubset(inspect.signature(launch_workshop).parameters):
    raise RuntimeError("Načteno staré Latium. Restartujte kernel/runtime a spusťte obě buňky znovu; první aktualizuje čistý workshop checkout.")
demo = launch_workshop(seed=SEED, covariance_drive_url=COVARIANCE_DRIVE_URL)


### Co je kde

- **První editace i vlastní fakta mají chat.** Oba chaty sdílejí historii i systémovou instrukci.
- GPU panel se aktualizuje každé dvě sekundy. Zdrojový kód je vpravo zbalený.
- Výchozí Qwen používá bf16 jako Latium main, fp32 deltu/Adam a vypnutý thinking.
- Nové otevření UI zachová model i aktivní editaci. **Obnovit původní model** je výslovná akce.
- **Pouze pro vývojáře PR:** `LOCAL_LATIUM` může ukázat na vlastní checkout větve;
  účastníci nechávají tuto volbu prázdnou. Existující repozitář se nepřepíná ani neresetuje.
- Pro soukromý Drive zapněte v první buňce `AUTHENTICATE_DRIVE` a přihlaste se před spuštěním Gradio.
- `COVARIANCE_DRIVE_URL` předvyplní odkaz v Gradio. Nový neprázdný odkaz jej nahradí;
  prázdná hodnota zachová poslední odkaz, případně výchozí. Stejný soubor se znovu nestahuje,
  změna ID stáhne novou matici při přípravě modelu. Odkaz lze změnit i přímo v Gradio před načtením.
- Čistý automatický checkout z GitHubu se při prvním spuštění kernelu aktualizuje přes fast-forward.
  Vlastní nebo rozpracovaný checkout se nemění; setup vypíše načtený commit.
- V Colabu otevřete vypsaný odkaz **`https://….gradio.live`**. Adresa runtime `*.colab.dev` může vrátit 404.
- Pokud máte načtenou starší implementaci s nefunkční proxy adresou, přidejte do volání
  `launch_workshop(...)` v druhé buňce `share=True` a spusťte ji znovu; živý model i edit se zachová.
- Když stránka ztratí spojení, ověřte runtime: pokud žije, znovu spusťte druhou buňku;
  pokud zanikl, spusťte obě a připravte model znovu. Limity Colabu tím neobcházíme.
- Pokud už kernel importoval jiné Latium nebo starou verzi balíčků, restartujte runtime.

První spuštění potřebuje internet pro balíčky, model a kovarianci. CPU regresní testy používají
malou skutečnou architekturu Qwen3.5; plný Qwen3.5-4B/T4 běh není v CPU prostředí ověřen.
